EJERCICIO 1


In [1]:
from math import exp,sqrt,log
from random import random
NV_MAGICCONST = 4 * exp(-0.5) / sqrt(2.0)
def normalvariate(mu, sigma):
  while 1:
    u1 = random()
    u2 = 1.0 - random()
    z = NV_MAGICCONST * (u1 - 0.5) / u2
    zz = z * z / 4.0
    if zz <= -log(u2):
      break
  return mu + z * sigma


def Media_Muestral_X(m,d):
#Estimaci´on de E[X] con desv´ıo est´andar d'
  Media = normalvariate(0,1)# X(1)
  Scuad, n = 0, 1 #Scuad = Sˆ2(1)
  while n <= m or sqrt(Scuad/n) > d:
    n += 1
    x = normalvariate(0,1)
    MediaAnt = Media
    Media = MediaAnt + (x - MediaAnt) / n
    Scuad = Scuad * (1 - 1 /(n-1)) + n*(Media - MediaAnt)**2
  return Media, n, Scuad

Media_Muestral_X(100, 0.1)

(-0.0813246571955297, 101, 0.9471361027397434)

EJERCICIO 2a

In [11]:
from random import random
import math
def g(u): ##funci´on a integrar en (0,1)
  return math.exp(u)/(math.sqrt(2*u))
##estimaci´on de la integral de g con Nsim simulaciones

def MonteCarlo1(g,m, d):
  Media = g(random())
  Scuad, n = 0, 1 #Scuad = Sˆ2(1)

  while n <= m or sqrt(Scuad/n) > d:
    n += 1
    x = g(random())
    MediaAnt = Media
    Media = MediaAnt + (x - MediaAnt) / n
    Scuad = Scuad * (1 - 1 /(n-1)) + n*(Media - MediaAnt)**2
  return Media, n, Scuad

def MonteCarlo(g, Nsim):
  Integral = 0
  for _ in range(Nsim):
    Integral += g(random())
  return Integral / Nsim

print(MonteCarlo1(g,100,0.01))
print(MonteCarlo(g, 100))

(2.0804094314036456, 35622, 3.562095190324644)
2.40503182515911


EJERCICIO 2b

In [23]:
def MonteCarlo_inf_inf1(f, m, d):
    u = random()
    Media = f(1 - 1/(1- u)) / (u**2)
    Scuad, n = 0, 1 #Scuad = Sˆ2(1)

    while n <= m or sqrt(Scuad/n) > d:
        n += 1
        u = random()
        x =f(1 - 1/(1- u)) / (u**2)

        MediaAnt = Media
        Media = MediaAnt + (x - MediaAnt) / n
        Scuad = Scuad * (1 - 1 /(n-1)) + n*(Media - MediaAnt)**2
    return Media, n, Scuad

def f (x):
    return (x ** 2 ) * exp(- x ** 2)

def MonteCarlo_inf_inf(func, N):
    suma_izq = 0
    suma_der = 0

    for _ in range(N):
        u = random()
        while u == 0:
            u = random()
        suma_izq += func(1 - 1/u) / (u**2)

    for _ in range(N):
        u = random()
        while u == 0:
            u = random()
        suma_der += func(1/u - 1) / (u**2)

    return suma_izq / N + suma_der / N


print(MonteCarlo_inf_inf1(f, 100, 0.01))
print(MonteCarlo_inf_inf(f,100))

(0.8831393749526447, 5030, 0.5029769396045924)
0.8529584025882972


EJERCICIO 3a



El intervalo de confianza para el valor esperado de una distribución estimado mediante la media muestral $\overline{X}(n)$ a partir de una muestra de tamaño $n > 100$ está dado por

$$\left(\overline{X}(n) - z_{\alpha/2}\frac{S(n)}{\sqrt{n}}, \,\, \overline{X}(n) + z_{\alpha/2}\frac{S(n)}{\sqrt{n}}\right)$$

*   Semi-ancho del intervalo: $ z_{\alpha/2}\frac{S(n)}{\sqrt{n}}$.

*   Ancho del intervalo:  $L = 2*z_{\alpha=0/2}\frac{S(n)}{\sqrt{n}}$


**Condiciones de parada:**

1) Si el semi-ancho pedido del IC es de $0.001$, quiero que la longitud de mi intervalo de confianza sea menor a $L=0.001*2=0.002$.
$$
2*z_{\alpha/2}\frac{S(n)}{\sqrt{n}} < L \quad ⇒ \quad \frac{S(n)}{\sqrt{n}} < \frac{L}{2*z_{\alpha/2}} = d
$$

2) Además, como estoy usando T.C.L., agrego la condición: $$n \geq 100.$$

In [ ]:

#CUANDO TENGO INTEGRALES DE 0 A 1
from scipy import stats
import math

def calculo_z(alpha):
  return stats.norm.ppf(1-alpha/2)

def ejercicio_2extra(fun, n_min, alpha, L):       # L = ancho del intervalo
  z_alpha_2 = calculo_z(alpha)
  d = L / (2 * z_alpha_2)
  media = fun(random())
  Scuad = 0
  n = 1
  while n<n_min or math.sqrt(Scuad/n) > d:               # mientras no se cumpla alguna de las condiciones mencionadas
      n = n + 1
      x = fun(random())           # Simular X
      media_ant = media
      media = media_ant + (x - media_ant) / n
      Scuad = Scuad * (1 - 1/(n-1)) + n * (media - media_ant)**2
  return media, Scuad, n


def intervalo(media, Scuad, alpha):
  z_alpha_2 = stats.norm.ppf(1-alpha/2)
  std = math.sqrt(Scuad/n)
  izq = media  -  z_alpha_2 * std
  der = media  +  z_alpha_2 * std
  intervalo = f"[{izq:.4f}, {der:.4f}]"
  return intervalo

n_min = 100
alpha = 0.05
L = 2 * 0.001
media, Scuad, n = ejercicio_2extra(g, n_min, alpha, L)

print('Valor real: ', 2.0685)
print('Estimación integral:', media)
print('Semi-ancho IC: ', stats.norm.ppf(1-alpha/2)*math.sqrt(Scuad/n))
print('Cantidad de datos generados N_s: ', n)
print('Intervalo de confianza: ', intervalo(media, Scuad, alpha))

Valor real:  2.0685
Estimación integral: 2.067845087388139
Semi-ancho IC:  0.000999999970408183
Cantidad de datos generados N_s:  21602212
Intervalo de confianza:  [2.0668, 2.0688]


In [41]:
def funcion(x):
    return math.sin(x)/x

def calculo_z(alpha):
  return stats.norm.ppf(1-alpha/2)

def ejercicio_2i(a, b,fun, n_min, alpha, L):       # L = ancho del intervalo
  z_alpha_2 = calculo_z(alpha)
  d = L / (2 * z_alpha_2)
  media = fun(a +(b - a) * random()) * (b-a)
  Scuad = 0
  n = 1
  while n<n_min or math.sqrt(Scuad/n) > d:               # mientras no se cumpla alguna de las condiciones mencionadas
      n = n + 1
      x = fun(a +(b - a) * random()) * (b-a)       # Simular X
      media_ant = media
      media = media_ant + (x - media_ant) / n
      Scuad = Scuad * (1 - 1/(n-1)) + n * (media - media_ant)**2
  return media, Scuad, n


def intervalo(media, Scuad, alpha):
  z_alpha_2 = stats.norm.ppf(1-alpha/2)
  std = math.sqrt(Scuad/n)
  izq = media  -  z_alpha_2 * std
  der = media  +  z_alpha_2 * std
  intervalo = f"[{izq:.4f}, {der:.4f}]"
  return intervalo

In [42]:
n_min = 100
alpha = 0.05
L = 2 * 0.001
media, Scuad, n = ejercicio_2i(math.pi, 2*math.pi, funcion, n_min, alpha, L)

from scipy import integrate
import math

def f(x):
    return math.sin(x) / x

resultado, error = integrate.quad(f, math.pi, 2*math.pi)
print('Valor real: ',resultado )
print('Estimación integral:', media)
print('Semi-ancho IC: ', stats.norm.ppf(1-alpha/2)*math.sqrt(Scuad/n))
print('Cantidad de datos generados N_s: ', n)
print('Intervalo de confianza: ', intervalo(media, Scuad, alpha))

Valor real:  -0.4337854758498377
Estimación integral: -0.4330103910435657
Semi-ancho IC:  0.0009999992896280656
Cantidad de datos generados N_s:  170033
Intervalo de confianza:  [-0.4340, -0.4320]


In [43]:
for N in [1000, 5000, 7000]:
    media, Scuad, n = ejercicio_2i(math.pi, 2*math.pi, funcion, N, alpha, L)
    print("N =", N)
    print('Intervalo de confianza: ', intervalo(media, Scuad, alpha))
    print(Scuad)
    print('Semi-ancho IC: ', stats.norm.ppf(1-alpha/2)*math.sqrt(Scuad/n))

N = 1000
Intervalo de confianza:  [-0.4346, -0.4326]
0.04433293763206799
Semi-ancho IC:  0.000999997517125083
N = 5000
Intervalo de confianza:  [-0.4345, -0.4325]
0.04444706996293526
Semi-ancho IC:  0.0009999987963209965
N = 7000
Intervalo de confianza:  [-0.4342, -0.4322]
0.04447519098611298
Semi-ancho IC:  0.0009999988724478852


EJERCICIO 3b

In [51]:
def calculo_z(alpha):
  return stats.norm.ppf(1-alpha/2)

def ejercicio_2ii(fun, n_min, alpha, L):       # L = ancho del intervalo
  z_alpha_2 = calculo_z(alpha)
  d = L / (2 * z_alpha_2)
  u = random()
  media = fun(1/u - 1) / (u ** 2)
  Scuad = 0
  n = 1
  while n<n_min or math.sqrt(Scuad/n) > d:               # mientras no se cumpla alguna de las condiciones mencionadas
      n = n + 1
      u = random()
      x  = fun(1/u - 1) / (u ** 2)     # Simular X
      media_ant = media
      media = media_ant + (x - media_ant) / n
      Scuad = Scuad * (1 - 1/(n-1)) + n * (media - media_ant)**2
  return media, Scuad, n


def intervalo(media, Scuad, alpha):
  z_alpha_2 = stats.norm.ppf(1-alpha/2)
  std = math.sqrt(Scuad/n)
  izq = media  -  z_alpha_2 * std
  der = media  +  z_alpha_2 * std
  intervalo = f"[{izq:.4f}, {der:.4f}]"
  return intervalo

In [52]:
def h(x):
    return 3/(3 + x **4)
n_min = 100
alpha = 0.05
L = 2 * 0.001
media, Scuad, n = ejercicio_2ii( h, n_min, alpha, L)
resultado, error = integrate.quad(h, 0, math.inf)
print('Valor real: ',resultado )
print('Estimación integral:', media)
print('Semi-ancho IC: ', stats.norm.ppf(1-alpha/2)*math.sqrt(Scuad/n))
print('Cantidad de datos generados N_s: ', n)
print('Intervalo de confianza: ', intervalo(media, Scuad, alpha))

Valor real:  1.4617906943750614
Estimación integral: 1.4627220122457318
Semi-ancho IC:  0.0009999999884711347
Cantidad de datos generados N_s:  3660791
Intervalo de confianza:  [1.4617, 1.4637]


EJERCICIO 4

# Ejercicio 4

Estimar $\pi$ sorteando puntos uniformemente distribuidos en el cuadrado cuyos vértices son: $(1,1)$, $(-1,1)$, $(-1,-1)$, $(1,-1)$, y contabilizando la fracción que cae dentro del círculo inscripto de radio $1$.


**a)** Dar la estimación de la proporción de puntos que caen dentro del círculo deteniendo la simulación cuando la desviación estándar muestral del estimador sea menor que $0.01$.

**b)** Construir un intervalo de confianza del $95\%$ para $\pi$ cuyo ancho sea menor que
$${\bf i)}\ 0.1, \quad {\bf ii)}\ 0.05, \quad {\bf iii)}\ 0.001$$
¿Cuántas simulaciones fueron necesarias en cada caso?


Queremos estimar el valor de $\pi$ utilizando simulación Monte Carlo.
Consideremos el cuadrado de lado $2$ centrado en el origen:
$[-1,1]\times[-1,1]$ y el círculo inscrito de radio $1$:
$x^2+y^2\leq 1$.

Si generamos puntos uniformemente distribuidos en el cuadrado, la probabilidad de que un punto caiga dentro del círculo es:

$$
p = \frac{\text{área círculo}}{\text{área cuadrado}}
= \frac{\pi r^2}{l^2}.
$$

Como $r=1$ y $l=2$,

$$
p=\frac{\pi}{4}
\qquad \Rightarrow \qquad
\pi = 4p.
$$

Definimos entonces la variable aleatoria:

$$
X_i=
\begin{cases}
1 & \text{si el punto cae dentro del círculo}\\
0 & \text{en otro caso}
\end{cases}
$$

Entonces:

$$
X_i \sim Bernoulli(p),
$$

y además:

$$
E[X_i]=p,
$$

$$
Var(X_i)=p(1-p).
$$

Sea:

$$
\overline{X}_n=\frac{1}{n}\sum_{i=1}^{n}X_i.
$$

Como $E[\overline{X}_n]=p$, un estimador para $\pi$ es:

$$
\hat{\pi}_n = 4\overline{X}_n.
$$

Además:

$$
Var(\overline{X}_n)
=
\frac{p(1-p)}{n}.
$$

Estimamos entonces su desvío estándar mediante:

$$
\sqrt{
\frac{\overline{X}_n(1-\overline{X}_n)}{n}
}.
$$

**a)**

Se simulan puntos uniformemente distribuidos en el cuadrado y se estima la proporción de puntos que caen dentro del círculo mediante:

$$
\hat{p}_n = \overline{X}_n.
$$

La simulación se detiene cuando el desvío estándar estimado del estimador satisface:

$$
\sqrt{
\frac{\overline{X}_n(1-\overline{X}_n)}{n}
}
< 0.01.
$$

In [54]:
import random

def en_el_circulo():
  u = random.uniform(-1,1)
  v = random.uniform(-1,1)
  if u**2+v**2 <= 1:
    return 1
  else:
    return 0

def ejercicio_6a(d):
  p = 0
  n = 0

  while n <= 100 or math.sqrt(p*(1-p)/n) > d:
    n = n + 1
    x = en_el_circulo()                 # Simular X
    p = p + (x-p) / n

  return p, n


d = 0.01
p, n = ejercicio_6a(d)

print('Ejercicio a:')
print('Estimación de proporción: ', p)
print('Estimación de pi: ', 4*p)    # Multiplico por 4 por el teórico de arriba
print('Número de simulaciones: ', n)

Ejercicio a:
Estimación de proporción:  0.7826086956521731
Estimación de pi:  3.1304347826086922
Número de simulaciones:  1702



**b)**

Queremos construir un intervalo de confianza del $95\%$ para $\pi$ de amplitud L menor que $0.1$.

Teníamos que un estimador para $\pi$ es: $\hat{\pi}_n = 4\overline{X}_n$, y  como $Var(\overline{X}_n) = \frac{p(1-p)}{n}$

obtenemos que:
$$
Var(\hat{\pi}_n)
=
16\frac{p(1-p)}{n}.
$$

Ahora, usando aproximación normal:
$$
\hat{\pi}_n
\pm
1.96\sqrt{
16\frac{\overline{X}_n(1-\overline{X}_n)}{n}
}.
$$

La amplitud del intervalo es:
$$
2\cdot1.96\cdot4
\sqrt{
\frac{\overline{X}_n(1-\overline{X}_n)}{n}
}.
$$

Se continúa simulando hasta que:
$$
2\cdot1.96\cdot4
\sqrt{
\frac{\overline{X}_n(1-\overline{X}_n)}{n}
}
< 0.1.
$$

de forma equivalente:

$$
\sqrt{
\frac{\overline{X}_n(1-\overline{X}_n)}{n}
}
< \frac{0.1}{2\cdot1.96\cdot4}.
$$


In [55]:
def ejercicio_6b(L=0.1, alpha=0.05):

    z_alpha_2 = stats.norm.ppf(1-alpha/2)
    cota = L / (4 * 2 * z_alpha_2)

    n = 0
    p = 0

    while n <= 100 or math.sqrt(p * (1-p) / n) > cota:
        n += 1
        X = en_el_circulo()
        p = p + (X - p) / n

    pi_est = 4 * p
    error = z_alpha_2 * 4 * math.sqrt(p * (1-p) / n)
    IC = (pi_est - error, pi_est + error)

    print(f"Estimación de pi = {pi_est}")
    print(f"Intervalo de confianza 95% = {IC}")
    print(f"Amplitud del intervalo maxima = {L}")
    print(f"Amplitud del intervalo = {2 * error}")
    print(f"Número de simulaciones = {n}")

    return pi_est, IC, n

In [56]:
print("Inciso b)")
print("\nAmplitud del IC < 0.1")
res1 = ejercicio_6b(L=0.1, alpha=0.05)

print("\nAmplitud del IC < 0.05")
res2 = ejercicio_6b(L=0.05, alpha=0.05)

print("\nAmplitud de IC < 0.001")
res3 = ejercicio_6b(L=0.001, alpha=0.05)

Inciso b)

Amplitud del IC < 0.1
Estimación de pi = 3.1185798816568093
Intervalo de confianza 95% = (3.0685874052490467, 3.168572358064572)
Amplitud del intervalo maxima = 0.1
Amplitud del intervalo = 0.09998495281552515
Número de simulaciones = 4225

Amplitud del IC < 0.05
Estimación de pi = 3.131881282910489
Intervalo de confianza 95% = (3.106882084514094, 3.1568804813068843)
Amplitud del intervalo maxima = 0.05
Amplitud del intervalo = 0.049998396792789744
Número de simulaciones = 16712

Amplitud de IC < 0.001
Estimación de pi = 3.1413903716587526
Intervalo de confianza 95% = (3.1408903716679966, 3.1418903716495086)
Amplitud del intervalo maxima = 0.001
Amplitud del intervalo = 0.0009999999815116095
Número de simulaciones = 41445163


In [ ]:
#YO

def valorPi(m,d):
    u = 2 * random() -1
    v = 2 * random() - 1
    if u ** 2 + v ** 2 <= 1:
      Media = 1
    else:
      Media = 0
    Scuad, n = 0, 1 #Scuad = Sˆ2(1)
    while n <= m or sqrt(Scuad/n) > d:
      n += 1
      u = 2 * random() -1
      v = 2 * random() - 1
      if u ** 2 + v ** 2 <= 1:
        x = 1
      else:
        x = 0
      MediaAnt = Media
      Media = MediaAnt + (x - MediaAnt) / n
      Scuad = Scuad * (1 - 1 /(n-1)) + n*(Media - MediaAnt)**2
    return Media, n, Scuad



valorPi(100,0.01)